# Blue Book for Bulldozers: faut-il spécialiser le modèle par famille ?

Estimer le prix d’un engin d’occasion exige de distinguer les caractéristiques de la machine et le contexte de sa vente. Ce notebook étudie une décision concrète : un CatBoost unique peut-il servir toutes les familles, ou des modèles spécialisés justifient-ils leur complexité supplémentaire ?

Le lecteur suit deux questions distinctes : **la segmentation aide-t-elle pour chacune des préparations A et B ?** Puis **la représentation plus détaillée des absences dans B apporte-t-elle quelque chose par rapport à A ?** Une seconde période vérifie ensuite la robustesse du choix.

La cible est `log1p(SalePrice)` ; le RMSE sur cette cible est le RMSLE des prix obtenus par `expm1`. Plus le score est faible, mieux c’est. Un gain sera toujours défini comme **score global − score segmenté** : positif, il favorise les spécialistes.

1. Fixer le protocole et séparer les périodes.
2. Expliquer et contrôler la préparation, sans apprendre de statistiques sur la validation.
3. Établir des références simples.
4. Comparer global/segmenté sous A, puis refaire la même comparaison sous B.
5. Distinguer l’effet de l’imputation de celui de l’architecture.
6. Confirmer sur une autre période.
7. Motiver le choix, critère par critère.
8. Expliciter la portée et les limites de la décision.

Tous les tableaux de résultats sont produits par les cellules exécutées. Les chiffres d'un prototype exploratoire antérieur, plus simple, ne sont pas ceux de ce notebook. Le test 2012 n’est pas évalué ; les anciennes versions sont conservées dans la sauvegarde externe.

## 1. Protocole fixé avant exécution

- **Validation principale :** septembre–décembre 2011 ; apprentissage jusqu’au 31 août 2011.
- **Confirmation :** janvier–avril 2011 ; apprentissage jusqu’au 31 décembre 2010. Cette fenêtre vérifie le passage à une année inédite et la même saison que le test 2012.
- **Modèles :** moyenne du log-prix globale et par segment, puis CatBoost global et segmenté. Paramètres communs fixés à 500 itérations, profondeur 6, learning rate 0,1, graine 2012. Aucune recherche Optuna, aucun early stopping sur la validation.
- **Imputation :** A utilise `Missing` ; B distingue les absences presque systématiques par segment. B n’est retenue que si son meilleur score principal améliore celui de A d’au moins 0,002. Sinon, priorité à A.
- **Architecture :** pour la méthode d’imputation retenue, exiger un gain segmenté principal d’au moins 0,003 RMSLE, une borne basse du bootstrap hebdomadaire supérieure à zéro et un gain strictement positif sur la confirmation. Sinon, retenir le global pour ce budget et documenter la segmentation comme non confirmée.

Ces règles de confirmation portent sur la préparation complète. Elles complètent une analyse exploratoire antérieure, sans en réinterpréter les résultats. Le seuil 0,002 est une tolérance pratique, pas une estimation du bruit. Le bootstrap ne couvre pas l’incertitude d’entraînement. Les sous-modèles disposent collectivement d’une capacité supérieure au global ; les budgets fixes ne prouvent pas une supériorité universelle de l’architecture.

Ne pas changer les règles ou les paramètres après lecture des scores sans identifier une nouvelle expérience exploratoire.

In [1]:
from pathlib import Path
from datetime import datetime
import sys
import json
import numpy as np
import pandas as pd
import catboost
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'preparation_03.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.preparation_03 import Preparation03
from src.experiences_02 import baselines, fit_pair, architecture_gain, rmsle_log

SEED = 2012
PARAMS = dict(iterations=500, depth=6, learning_rate=0.1,
              loss_function='RMSE', random_seed=SEED,
              thread_count=8,  # fixé : rend les durées comparables d'une exécution à l'autre
              allow_writing_files=False, verbose=100)
RUN_DIR = ROOT / 'resultats' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)

In [2]:
RAW = ROOT / 'data' / 'raw' / 'bluebook-for-bulldozers' / 'TrainAndValid.csv'
raw = pd.read_csv(RAW, low_memory=False)
raw['saledate'] = pd.to_datetime(raw['saledate'], format='%m/%d/%Y %H:%M')
# Le CSV est combiné : exclure 2012 avant toute analyse ou transformation de cible.
raw = raw.loc[raw.saledate < '2012-01-01'].sort_values('saledate', kind='stable').reset_index(drop=True)
assert raw.saledate.is_monotonic_increasing and raw.saledate.max() < pd.Timestamp('2012-01-01')
assert raw.SalePrice.gt(0).all()
train_main = raw.loc[raw.saledate < '2011-09-01'].copy()
val_main = raw.loc[raw.saledate >= '2011-09-01'].copy()
train_confirmation = raw.loc[raw.saledate < '2011-01-01'].copy()
val_confirmation = raw.loc[(raw.saledate >= '2011-01-01') & (raw.saledate < '2011-05-01')].copy()
windows = {'principal': (train_main, val_main), 'confirmation': (train_confirmation, val_confirmation)}
display(pd.DataFrame([dict(fenetre=k, n_train=len(t), n_validation=len(v),
                         fin_train=t.saledate.max(), debut_validation=v.saledate.min(),
                         fin_validation=v.saledate.max()) for k, (t, v) in windows.items()]))
# Conditions de l'expérience, relues avec les scores : versions, graine, budget et règles.
manifest = dict(seed=SEED, params=PARAMS, pandas=pd.__version__, numpy=np.__version__,
                catboost=catboost.__version__, n_train_total=len(raw),
                rules={'gain_B_min': .002, 'gain_seg_min': .003,
                       'bootstrap_lower_gt': 0, 'confirmation_gain_gt': 0})
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Conditions de l\'expérience enregistrées dans', RUN_DIR / 'manifest.json')

,fenetre,n_train,n_validation,fin_train,debut_validation,fin_validation
0,principal,390213,10912,2011-08-31,2011-09-01,2011-12-30
1,confirmation,365928,12549,2010-12-30,2011-01-04,2011-04-30


Conditions de l'expérience enregistrées dans c:\Users\Abram\Desktop\Fiche\Projet\resultats\20260911_214318_590008\manifest.json


## 2. Préparation complète et corrections explicites

La classe `Preparation03` dans `src/preparation_03.py` expose `fit` et `transform`. Les deux architectures utilisent **la même préparation ajustée sur le même apprentissage**, y compris les fréquences globales ; les sous-modèles retirent seulement la colonne constante `ProductGroup`.

| Élément | Traitement |
|---|---|
| Identifiants | Suppression de SalesID, MachineID et ModelID |
| Descriptions | fiModelDesc et fiProductClassDesc conservées pour CatBoost, en plus des composantes existantes |
| Spécifications | Quatre colonnes numériques : chevaux, tonnes, profondeur, capacité ; centre d’intervalle ou borne basse ouverte |
| Heures | Zéro, négatif et >40 000 → manquant ; indicateur hours_reported conservé |
| Année | <1950 ou postérieure à l’année de vente → manquante |
| Variables dérivées | Année, mois, jour, trimestre, âge, heures/an, récession 2008–2009, préfixe du modèle |
| Fréquence du modèle | Comptages appris sur l’apprentissage ; inédit → 0 |
| Hydraulics | Modalités de fréquence >500 apprises sur l’apprentissage ; autres → Other |
| Année/heures manquantes | Médiane par ProductGroup si ≥30 observations, sinon globale ; indicateurs de manque |
| Variables dérivées manquantes | Sentinelle -1, comme dans 03 ; pas de ratio artificiel calculé après imputation |
| Catégories | Chaînes natives, sans encodage ordinal ; Missing pour les absences |

Le seuil 30 est une convention de repli. Les bornes et milieux d’intervalles sont des approximations déclarées. NO ROPS reste distinct d’OROPS ; EROPS AC est harmonisé avec EROPS w AC. Les descriptions originales sont gardées pour ne pas supprimer l’information « Compact Construction » des classes non identifiées.

**Évolutions par rapport à une version archivée antérieure de ce notebook :** isolation de la préparation, tri effectif des lignes, borne d’année relative à la vente, conservation des descriptions et catégories natives. Ce notebook ne mesure pas séparément l’effet de chacune de ces corrections.

In [3]:
preparation = Preparation03('A').fit(train_main)
X_inspection = preparation.transform(val_main.head(32))
assert preparation.fit_max_date_ < val_main.saledate.min()
assert not {'SalePrice', 'logSalePrice', 'saledate'} & set(X_inspection.columns)
assert X_inspection[preparation.cat_features_].notna().all().all()
print(f'{len(preparation.feature_names_)} features dont {len(preparation.cat_features_)} catégorielles.')
print(f'Statistiques ajustées sur {preparation.n_fit_:,} lignes, jusqu’au {preparation.fit_max_date_.date()}.')
print('Âges négatifs observés avant nettoyage, apprentissage :',
      int((train_main.YearMade > train_main.saledate.dt.year).sum()))

63 features dont 46 catégorielles.
Statistiques ajustées sur 390,213 lignes, jusqu’au 2011-08-31.
Âges négatifs observés avant nettoyage, apprentissage : 12


### 2.1. A/B : contrôle de l’hypothèse d’absence

A représente toutes les catégories manquantes par `Missing`. B apprend les profils d’absence sur l’apprentissage, avec le diagnostic déjà réalisé plus haut dans ce notebook (seuils 5/95 % et tolérance de 5 % dans la zone intermédiaire). Elle ajoute les champs ProductSize, Forks, Ride_Control, Transmission et Coupler au contrôle par segment.

Dans ces champs, une absence >95 % dans un segment reçoit `Absent dans ce segment` ; les autres absences restent `Missing`. Le libellé décrit une convention statistique, pas une non-applicabilité métier démontrée. Les règles sont figées à transform : la composition de la validation ne les modifie pas.

## 3. Mesurer une référence simple avant CatBoost

Une prédiction élaborée doit apporter quelque chose face à une règle simple. La moyenne du log-prix globale attribue la même prédiction à toutes les machines ; la moyenne par famille ajoute seulement `ProductGroup`. Ces références sont calculées exclusivement sur l’apprentissage de chaque période.

Les deux scores permettent de distinguer ce qu’apporte déjà la famille de ce qu’apprennent ensuite les modèles à partir de toutes les caractéristiques. Ils seront comparés aux scores CatBoost de la même période.

In [4]:
baseline_rows = []
for label, (train, validation) in windows.items():
    y = np.log1p(validation.SalePrice)
    for name, prediction in baselines(train, validation).items():
        score = rmsle_log(y, prediction)
        print(f'{label} — {name} : RMSLE validation = {score:.5f}', flush=True)
        baseline_rows.append(dict(fenetre=label, modele=name, rmsle=score))
baseline_scores = pd.DataFrame(baseline_rows)
baseline_scores.to_csv(RUN_DIR/'baselines.csv', index=False)
display(baseline_scores)

principal — Moyenne log globale : RMSLE validation = 0.72928
principal — Moyenne log par segment : RMSLE validation = 0.60851
confirmation — Moyenne log globale : RMSLE validation = 0.71424
confirmation — Moyenne log par segment : RMSLE validation = 0.60453


,fenetre,modele,rmsle
0,principal,Moyenne log globale,0.729278
1,principal,Moyenne log par segment,0.608507
2,confirmation,Moyenne log globale,0.714244
3,confirmation,Moyenne log par segment,0.604527


## 4. Comparer la segmentation, d’abord sous A puis sous B

Le global reçoit `ProductGroup` : il connaît donc déjà la famille de la machine. La segmentation ne lui ajoute pas cette information ; elle impose un apprentissage spécialisé. Les features, les données, les réglages CatBoost et les statistiques de préparation restent communs aux deux architectures. Seule la colonne `ProductGroup`, constante à l’intérieur d’un spécialiste, est retirée de ses entrées.

Chaque spécialiste nécessite au moins 200 ventes d’apprentissage. Sinon, ou pour une famille inédite, le global sert de repli. Le score agrégé couvre toutes les ventes de validation : aucune ligne difficile n’est exclue.

Les scores s’affichent à la fin du global, de chaque spécialiste et de l’ensemble segmenté. Les tableaux restent visibles sous le bloc A avant de lancer B, puis sous B. Le score du global affiché pour une famille est recalculé sur **les mêmes ventes de cette famille** : il ne faut pas comparer le score d’un spécialiste au score global toutes familles confondues.

Les 500 itérations sont fixées à l’avance ; la progression CatBoost concerne l’apprentissage, pas une optimisation sur cette validation. Le temps segmenté affiché couvre les spécialistes ; le global de repli a été entraîné séparément.

### 4.1. Préparation A, global contre segmenté

A code les catégories absentes par `Missing`. Cette préparation simple sert de point de départ pour mesurer l’intérêt de la spécialisation.

**À lire :** le score global, le score segmenté, le gain et sa variabilité hebdomadaire, puis les gains/pertes par famille. Un gain agrégé ne signifie pas que chaque famille gagne.

In [5]:
results = {}
main_gains = {}

results['A'] = fit_pair(train_main, val_main, 'A', PARAMS,
                             output_dir=RUN_DIR, label='principal')
print('\nTABLEAU A — même validation, comparaison des deux architectures')
display(results['A']['scores'])
main_gains['A'] = architecture_gain(results['A'], seed=SEED)
print('GAIN DE SEGMENTATION A — positif = favorable aux spécialistes')
display(pd.DataFrame([main_gains['A']]))
print('DÉTAIL A — global et spécialiste évalués sur les mêmes lignes par famille')
display(results['A']['segments'])


[principal — préparation A]
Apprentissage : 390,213 ventes jusqu'au 2011-08-31 ; validation : 10,912 ventes du 2011-09-01 au 2011-12-30.
63 variables, statistiques figées sur l'apprentissage.
0:	learn: 0.6487552	total: 938ms	remaining: 7m 47s
100:	learn: 0.2529541	total: 1m 48s	remaining: 7m 8s
200:	learn: 0.2361007	total: 3m 34s	remaining: 5m 18s
300:	learn: 0.2281017	total: 5m 19s	remaining: 3m 31s
400:	learn: 0.2227735	total: 7m 16s	remaining: 1m 47s
499:	learn: 0.2191599	total: 9m 39s	remaining: 0us
GLOBAL - RMSLE validation = 0.23351 (586 s)
  spécialiste BL : 77,642 lignes
0:	learn: 0.3673723	total: 181ms	remaining: 1m 30s
100:	learn: 0.1622430	total: 17.2s	remaining: 1m 8s
200:	learn: 0.1525001	total: 34.7s	remaining: 51.6s
300:	learn: 0.1482600	total: 52.2s	remaining: 34.5s
400:	learn: 0.1452622	total: 1m 9s	remaining: 17.2s
499:	learn: 0.1432418	total: 1m 26s	remaining: 0us
  spécialiste MG : 24,841 lignes
0:	learn: 0.7015063	total: 99.4ms	remaining: 49.6s
100:	learn: 0.20610

,fenetre,methode,architecture,rmsle,secondes,replis
0,principal,A,global,0.233509,586.181358,0
1,principal,A,segmente,0.217395,486.526761,0


GAIN DE SEGMENTATION A — positif = favorable aux spécialistes


,gain,borne_basse_95,borne_haute_95,semaines
0,0.016114,0.012559,0.020032,18


DÉTAIL A — global et spécialiste évalués sur les mêmes lignes par famille


,segment,n,global,segmente,gain,replis
0,BL,1773,0.177801,0.170024,0.007776,0
1,MG,648,0.233100,0.205751,0.027349,0
2,SSL,1567,0.203020,0.178505,0.024515,0
3,TEX,3164,0.240799,0.226502,0.014297,0
4,TTT,1965,0.259021,0.243661,0.015360,0
5,WL,1795,0.262673,0.245408,0.017264,0


### 4.2. Préparation B, global contre segmenté

B distingue certaines absences presque systématiques par famille. On répète exactement la comparaison précédente : les tableaux montrent si le gain de segmentation se maintient avec cette autre représentation.

**À lire :** le score global, le score segmenté, le gain et sa variabilité hebdomadaire, puis les gains/pertes par famille. Un gain agrégé ne signifie pas que chaque famille gagne.

In [6]:
results['B'] = fit_pair(train_main, val_main, 'B', PARAMS,
                             output_dir=RUN_DIR, label='principal')
print('\nTABLEAU B — même validation, comparaison des deux architectures')
display(results['B']['scores'])
main_gains['B'] = architecture_gain(results['B'], seed=SEED)
print('GAIN DE SEGMENTATION B — positif = favorable aux spécialistes')
display(pd.DataFrame([main_gains['B']]))
print('DÉTAIL B — global et spécialiste évalués sur les mêmes lignes par famille')
display(results['B']['segments'])


[principal — préparation B]
Apprentissage : 390,213 ventes jusqu'au 2011-08-31 ; validation : 10,912 ventes du 2011-09-01 au 2011-12-30.
63 variables, statistiques figées sur l'apprentissage.
0:	learn: 0.6487552	total: 880ms	remaining: 7m 19s
100:	learn: 0.2527525	total: 2m 10s	remaining: 8m 36s
200:	learn: 0.2358776	total: 4m 47s	remaining: 7m 7s
300:	learn: 0.2278147	total: 7m 15s	remaining: 4m 48s
400:	learn: 0.2224089	total: 9m 56s	remaining: 2m 27s
499:	learn: 0.2190138	total: 12m 41s	remaining: 0us
GLOBAL - RMSLE validation = 0.23258 (768 s)
  spécialiste BL : 77,642 lignes
0:	learn: 0.3673723	total: 185ms	remaining: 1m 32s
100:	learn: 0.1622430	total: 18.2s	remaining: 1m 12s
200:	learn: 0.1525001	total: 36.7s	remaining: 54.6s
300:	learn: 0.1482600	total: 55.4s	remaining: 36.7s
400:	learn: 0.1452622	total: 1m 14s	remaining: 18.4s
499:	learn: 0.1432418	total: 1m 33s	remaining: 0us
  spécialiste MG : 24,841 lignes
0:	learn: 0.7015063	total: 115ms	remaining: 57.4s
100:	learn: 0.206

,fenetre,methode,architecture,rmsle,secondes,replis
0,principal,B,global,0.232582,768.209319,0
1,principal,B,segmente,0.217395,536.056944,0


GAIN DE SEGMENTATION B — positif = favorable aux spécialistes


,gain,borne_basse_95,borne_haute_95,semaines
0,0.015187,0.012048,0.018641,18


DÉTAIL B — global et spécialiste évalués sur les mêmes lignes par famille


,segment,n,global,segmente,gain,replis
0,BL,1773,0.177780,0.170024,0.007756,0
1,MG,648,0.228734,0.205751,0.022982,0
2,SSL,1567,0.202938,0.178505,0.024433,0
3,TEX,3164,0.240629,0.226502,0.014127,0
4,TTT,1965,0.256976,0.243661,0.013315,0
5,WL,1795,0.261598,0.245408,0.016190,0


## 5. Séparer l’effet A/B de l’effet global/segmenté

Le premier tableau réunit les quatre candidats. Le second compare **B à A à architecture constante** : il évite d’attribuer à l’imputation un gain qui viendrait seulement de la segmentation. Les gains global/segmenté de A et B sont également récapitulés.

La règle de sélection reste celle fixée au départ : comparer le meilleur candidat A au meilleur candidat B, puis ne retenir B qu’à partir de 0,002 de gain. Cette sélection conjointe n’est pas une preuve que B est meilleur dans toutes les architectures. Les tableaux servent à expliciter cette nuance, sans modifier le critère après observation.

In [7]:
main_scores = pd.concat([results[m]['scores'] for m in ['A', 'B']], ignore_index=True)
print('LES QUATRE CANDIDATS — validation principale')
display(main_scores)
main_scores.to_csv(RUN_DIR/'comparaison_principale.csv', index=False)
by_architecture = main_scores.pivot(index='architecture', columns='methode', values='rmsle')
by_architecture['gain_B_sur_A'] = by_architecture['A'] - by_architecture['B']
print('EFFET DE L’IMPUTATION — à architecture constante ; positif = B meilleur')
display(by_architecture)
print('EFFET DE LA SEGMENTATION — à imputation constante ; positif = segmenté meilleur')
display(pd.DataFrame(main_gains).T.rename_axis('methode'))
by_architecture.to_csv(RUN_DIR/'comparaison_imputation.csv')

LES QUATRE CANDIDATS — validation principale


,fenetre,methode,architecture,rmsle,secondes,replis
0,principal,A,global,0.233509,586.181358,0
1,principal,A,segmente,0.217395,486.526761,0
2,principal,B,global,0.232582,768.209319,0
3,principal,B,segmente,0.217395,536.056944,0


EFFET DE L’IMPUTATION — à architecture constante ; positif = B meilleur


methode,A,B,gain_B_sur_A
architecture,,,
global,0.233509,0.232582,0.000927
segmente,0.217395,0.217395,0.000000


EFFET DE LA SEGMENTATION — à imputation constante ; positif = segmenté meilleur


,gain,borne_basse_95,borne_haute_95,semaines
methode,,,,
A,0.016114,0.012559,0.020032,18.0
B,0.015187,0.012048,0.018641,18.0


### Pourquoi les deux scores segmentés sont-ils identiques ?

Le tableau ci-dessus n'est pas une erreur de recopie : `segmenté A` et `segmenté B` affichent la **même** valeur, jusqu'à la dernière décimale, et famille par famille.

C'est une conséquence directe de la façon dont B est construite. Un spécialiste n'est entraîné que sur **une seule** famille. Or, à l'intérieur d'une famille donnée, la règle B ne change pas quelles lignes sont manquantes : elle remplace simplement l'étiquette `Missing` par `Absent dans ce segment` sur ces mêmes lignes. La partition des observations est identique, seul le libellé diffère.

CatBoost traite une variable catégorielle par les statistiques de cible associées à chaque modalité, pas par le texte de la modalité. Renommer une catégorie sans modifier les lignes qu'elle regroupe produit donc exactement le même arbre, et exactement les mêmes prédictions.

**Ce que cela implique pour la lecture des résultats :** l'effet de la méthode B ne peut se manifester que dans le modèle global, seul endroit où plusieurs familles coexistent et où le libellé distingue réellement deux situations. C'est ce que montre le tableau à architecture constante de la cellule suivante.

In [8]:
best_A = results['A']['scores'].rmsle.min()
best_B = results['B']['scores'].rmsle.min()
gain_B = float(best_A - best_B)
method_selected = 'B' if gain_B >= .002 else 'A'
print(f'Meilleur A : {best_A:.5f} ; meilleur B : {best_B:.5f}.')
print(f'Gain B sur A : {gain_B:+.5f} ; seuil requis pour retenir B : 0,002.')
print('Méthode fixée pour la confirmation :', method_selected)
print('B dépasse le seuil pratique.' if method_selected == 'B'
      else 'Le seuil de gain requis pour B n’est pas atteint : priorité à la simplicité de A.')
main_gain = main_gains[method_selected]
print('L’architecture attend encore le contrôle sur la seconde période.')

Meilleur A : 0.21740 ; meilleur B : 0.21740.
Gain B sur A : +0.00000 ; seuil requis pour retenir B : 0,002.
Méthode fixée pour la confirmation : A
Le seuil de gain requis pour B n’est pas atteint : priorité à la simplicité de A.
L’architecture attend encore le contrôle sur la seconde période.


## 6. Vérifier le gain sur janvier–avril d’une année inédite

La comparaison principale peut dépendre de la conjoncture ou de la saison. On entraîne donc à nouveau avant 2011, puis on prédit janvier–avril 2011 : l’année à prédire n’apparaît pas dans l’apprentissage. La préparation est elle aussi réajustée sur ce seul passé.

La méthode A/B est déjà fixée par le bloc précédent. **On compare à nouveau global et segmenté pour cette méthode seulement**, avec les mêmes hyperparamètres. Le résultat ne doit pas servir à inventer une nouvelle règle d’imputation après coup.

À lire : le signe du gain, les familles qui gagnent ou perdent et les éventuels replis. Cette fenêtre appartient à l’historique d’apprentissage du modèle principal : c’est un contrôle temporel complémentaire, pas un test indépendant. Aucune donnée de 2012 n’intervient.

In [9]:
print(f'CONFIRMATION — préparation {method_selected}, global contre segmenté')
confirmation = fit_pair(train_confirmation, val_confirmation, method_selected, PARAMS,
                        output_dir=RUN_DIR, label='confirmation')
confirmation_gain = architecture_gain(confirmation, seed=SEED)
print('COMPARAISON SUR LA SECONDE PÉRIODE')
display(confirmation['scores'])
print('GAIN ET VARIABILITÉ HEBDOMADAIRE — descriptifs')
display(pd.DataFrame([confirmation_gain]))
print('QUELLES FAMILLES CONFIRMENT LE GAIN ?')
display(confirmation['segments'])

CONFIRMATION — préparation A, global contre segmenté

[confirmation — préparation A]
Apprentissage : 365,928 ventes jusqu'au 2010-12-30 ; validation : 12,549 ventes du 2011-01-04 au 2011-04-30.
63 variables, statistiques figées sur l'apprentissage.
0:	learn: 0.6493007	total: 1.29s	remaining: 10m 42s
100:	learn: 0.2523731	total: 2m 25s	remaining: 9m 36s
200:	learn: 0.2356984	total: 4m 54s	remaining: 7m 17s
300:	learn: 0.2273464	total: 7m 31s	remaining: 4m 58s
400:	learn: 0.2225666	total: 9m 55s	remaining: 2m 27s
499:	learn: 0.2189509	total: 12m 21s	remaining: 0us
GLOBAL - RMSLE validation = 0.24370 (753 s)
  spécialiste BL : 72,993 lignes
0:	learn: 0.3679817	total: 171ms	remaining: 1m 25s
100:	learn: 0.1618150	total: 17.5s	remaining: 1m 9s
200:	learn: 0.1523518	total: 33.3s	remaining: 49.5s
300:	learn: 0.1480597	total: 51.1s	remaining: 33.8s
400:	learn: 0.1452910	total: 1m 8s	remaining: 17s
499:	learn: 0.1430922	total: 1m 26s	remaining: 0us
  spécialiste MG : 23,324 lignes
0:	learn: 0.6

,fenetre,methode,architecture,rmsle,secondes,replis
0,confirmation,A,global,0.243701,752.793444,0
1,confirmation,A,segmente,0.230495,510.899344,0


GAIN ET VARIABILITÉ HEBDOMADAIRE — descriptifs


,gain,borne_basse_95,borne_haute_95,semaines
0,0.013206,0.01072,0.016135,17


QUELLES FAMILLES CONFIRMENT LE GAIN ?


,segment,n,global,segmente,gain,replis
0,BL,2430,0.178576,0.166540,0.012036,0
1,MG,880,0.238705,0.215490,0.023216,0
2,SSL,1513,0.225562,0.207530,0.018032,0
3,TEX,3223,0.273572,0.265652,0.007920,0
4,TTT,2267,0.247953,0.234284,0.013669,0
5,WL,2236,0.267570,0.251469,0.016102,0


## 7. Motiver le choix, critère par critère

La décision ci-dessous applique les trois conditions préspécifiées à la préparation retenue. Le tableau indique lesquelles sont satisfaites ; un gain positif seul ne suffit pas à justifier six spécialistes. Si une condition manque, le choix global signifie « segmentation non confirmée dans ce protocole », pas « segmentation toujours inutile ».

La lecture des tableaux par famille reste nécessaire pour exposer les limites du choix agrégé. Cette cellule conclut sur les validations internes, sans prétendre fournir une performance finale sur 2012.

In [10]:
segmented_confirmed = (main_gain['gain'] >= .003
                       and main_gain['borne_basse_95'] > 0
                       and confirmation_gain['gain'] > 0)
architecture_selected = 'segmente' if segmented_confirmed else 'global'
decision = dict(methode=method_selected, architecture=architecture_selected,
                gain_imputation_B=gain_B, principal=main_gain,
                confirmation=confirmation_gain, params=PARAMS,
                statut='choix sur validations internes, test 2012 non évalué')
(RUN_DIR/'decision.json').write_text(json.dumps(decision, ensure_ascii=False, indent=2), encoding='utf-8')
criteria = pd.DataFrame([
    dict(critere='Gain principal', mesure=main_gain['gain'], regle='≥ 0,003',
         satisfait=main_gain['gain'] >= .003),
    dict(critere='Borne basse hebdomadaire principale', mesure=main_gain['borne_basse_95'],
         regle='> 0', satisfait=main_gain['borne_basse_95'] > 0),
    dict(critere='Gain de confirmation', mesure=confirmation_gain['gain'], regle='> 0',
         satisfait=confirmation_gain['gain'] > 0),
])
print('VÉRIFICATION DES CRITÈRES ÉCRITS AVANT CALCUL')
display(criteria)
criteria.to_csv(RUN_DIR/'criteres_decision.csv', index=False)
print(f'Choix selon le protocole : CatBoost {architecture_selected}, préparation {method_selected}.')
print('Gain principal :', round(main_gain['gain'], 5))
print('Gain confirmation :', round(confirmation_gain['gain'], 5))
print('Résultats enregistrés dans', RUN_DIR)

VÉRIFICATION DES CRITÈRES ÉCRITS AVANT CALCUL


,critere,mesure,regle,satisfait
0,Gain principal,0.016114,"≥ 0,003",True
1,Borne basse hebdomadaire principale,0.012559,> 0,True
2,Gain de confirmation,0.013206,> 0,True


Choix selon le protocole : CatBoost segmente, préparation A.
Gain principal : 0.01611
Gain confirmation : 0.01321
Résultats enregistrés dans c:\Users\Abram\Desktop\Fiche\Projet\resultats\20260911_214318_590008


## 8. Portée de la décision

Le choix obtenu concerne cette préparation et ce budget fixe de CatBoost. Il ne démontre ni que le modèle global optimisé ne pourrait pas rattraper la segmentation, ni que toutes les périodes futures suivront le même comportement. Les intervalles calculés ne reflètent que la variabilité hebdomadaire des erreurs sur ces validations, pas celle des entraînements.

Une seule graine est utilisée. Les prix de 2012 ont historiquement été consultés dans des versions précédentes du projet : l’exclusion actuelle prévient de nouvelles consultations sans effacer ce passé. Les comparaisons avec le leaderboard Kaggle ne sont pas pertinentes pour les scores de validation interne de ce notebook.

**Étape suivante après lecture des résultats :** arrêter la configuration à publier, puis effectuer séparément l’évaluation finale sur 2012. Ce notebook n’entraîne ni n’évalue automatiquement sur le test et ne remplit aucun score final à la main.